# Cross-encoder fine-tuning and scoring (D-028) — Amazon ML Challenge 2026, business entity resolution

Model: `intfloat/multilingual-e5-small` (MIT license, 117.7M parameters), run locally in this notebook (no API),
fine-tuned **only on the provided train data** (the bundle's `train.parquet`, FIT group of our pipeline).

1. Runtime → Change runtime type → **T4 GPU**.
2. Upload the bundle folder `ber_ce/` (train.parquet, score_tune.parquet, score_holdout.parquet, score_test.parquet)
   to **My Drive** (keep it private).
3. Runtime → **Run all**. Results are written to `My Drive/ber_ce/out/` (ce_tune/holdout/test.parquet).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, time, math, numpy as np, pandas as pd, torch
BUNDLE = '/content/drive/MyDrive/ber_ce'
OUT = BUNDLE + '/out'
os.makedirs(OUT, exist_ok=True)
print(os.listdir(BUNDLE))
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
BASE, MAXLEN, SEED = 'intfloat/multilingual-e5-small', 64, 42
EPOCHS, BS, LR = 2, 128, 4e-5
torch.manual_seed(SEED); np.random.seed(SEED)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1).cuda()
tr = pd.read_parquet(f'{BUNDLE}/train.parquet')
tr = tr.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_dev = 10000
dev, tr = tr.iloc[:n_dev].reset_index(drop=True), tr.iloc[n_dev:].reset_index(drop=True)
print('train', len(tr), 'pos rate', tr.label.mean(), 'dev', len(dev))

In [ ]:
def batches(df, bs, shuffle, seed=0):
    idx = np.random.default_rng(seed).permutation(len(df)) if shuffle else np.arange(len(df))
    for s in range(0, len(df), bs):
        j = idx[s:s + bs]
        enc = tok(df.ta.values[j].tolist(), df.tb.values[j].tolist(), truncation=True, max_length=MAXLEN,
                  padding=True, return_tensors='pt')
        yield j, {k: v.cuda(non_blocking=True) for k, v in enc.items()}

def evaluate(df):
    model.eval(); out = np.zeros(len(df), dtype=np.float32)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for j, enc in batches(df, 1024, False):
            out[j] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    model.train(); return out

steps = EPOCHS * math.ceil(len(tr) / BS)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 500) * max(0.0, 1 - s / steps))
scaler = torch.amp.GradScaler('cuda')
y_all = torch.tensor(tr.label.values, dtype=torch.float32)
step, t0 = 0, time.time()
model.train()
for ep in range(EPOCHS):
    for j, enc in batches(tr, BS, True, seed=SEED + ep):
        with torch.autocast('cuda', dtype=torch.float16):
            logit = model(**enc).logits.squeeze(-1)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logit.float(), y_all[j].cuda())
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
        if step % 500 == 0:
            el = time.time() - t0
            print(f'epoch {ep} step {step}/{steps} loss {loss.item():.4f} {el/60:.1f} min, eta {el/step*(steps-step)/60:.1f} min', flush=True)
    d = evaluate(dev); p = 1 / (1 + np.exp(-d)); yv = dev.label.values
    ll = -np.mean(yv * np.log(np.clip(p, 1e-7, 1)) + (1 - yv) * np.log(np.clip(1 - p, 1e-7, 1)))
    print(f'epoch {ep} dev logloss {ll:.4f} acc {np.mean((p > 0.5) == yv):.4f}', flush=True)
model.save_pretrained(f'{OUT}/model'); tok.save_pretrained(f'{OUT}/model')

In [ ]:
for g in ('tune', 'holdout', 'test'):
    df = pd.read_parquet(f'{BUNDLE}/score_{g}.parquet')
    order = np.argsort((df.ta.str.len() + df.tb.str.len()).values, kind='stable')
    df = df.iloc[order].reset_index(drop=True)
    t0 = time.time(); ce = evaluate(df)
    pd.DataFrame({'lid': df.lid, 'rid': df.rid, 'ce': ce}).to_parquet(f'{OUT}/ce_{g}.parquet', index=False)
    print(g, len(df), f'{(time.time()-t0)/60:.1f} min', flush=True)
print('DONE — download My Drive/ber_ce/out/ce_tune.parquet, ce_holdout.parquet, ce_test.parquet')